# 0605 디펜스 피드백

## error case 분석

### 모든 모델이 공통적으로 틀린 문제

In [9]:
# 모든 모델에서 모두 틀린 케이스
import pandas as pd
# 1. gemma
result_df = pd.read_csv("/workspace/data/test_set_process/inference/wsd_set_entire_labeled_ambiguous_sentence_sense_search3_gemma-3-27b-it.csv")

for index, row in result_df.iterrows():
    pred_label = row["generated_text"].split("A:")[-1].strip().lower()
    mapped_label = int(pred_label)
    result_df.at[index, "predicted_label"] = mapped_label

gemma_wrong_cases = result_df[result_df["predicted_label"] != result_df["gold_sense"]]
gemma_wrong_cases = set(gemma_wrong_cases["word_index"].tolist())

# 2. qwen3
result_df = pd.read_csv("/workspace/data/test_set_process/inference/wsd_set_entire_labeled_ambiguous_sentence_sense_search4_qwen3-vl-30b-instruct.csv")

for index, row in result_df.iterrows():
    pred_label = row["generated_text"].split("A:")[-1].strip().lower()
    try:
        mapped_label = int(pred_label)
    except ValueError:
        mapped_label = -1  # 예외 처리: 정수로 변환할 수 없는 경우 -1로 설정
    result_df.at[index, "predicted_label"] = mapped_label

qwen_wrong_cases = result_df[result_df["predicted_label"] != result_df["gold_sense"]]
qwen_wrong_cases = set(qwen_wrong_cases["word_index"].tolist())

# 3. mistral-small-3.1
result_df = pd.read_csv("/workspace/data/test_set_process/inference/wsd_set_entire_labeled_ambiguous_sentence_sense_search3_mistral-small-3.csv")

for index, row in result_df.iterrows():
    pred_label = row["generated_text"].split("A:")[-1].strip().lower()
    try:
        mapped_label = int(pred_label)
    except ValueError:
        mapped_label = -1  # 예외 처리: 정수로 변환할 수 없는 경우 -1로 설정
    result_df.at[index, "predicted_label"] = mapped_label

mistral_wrong_cases = result_df[result_df["predicted_label"] != result_df["gold_sense"]]
mistral_wrong_cases = set(mistral_wrong_cases["word_index"].tolist())

# 세 집합의 교집합
all_wrong_cases = gemma_wrong_cases.intersection(qwen_wrong_cases).intersection(mistral_wrong_cases)
print("모든 모델에서 틀린 케이스의 word_index:", all_wrong_cases)

모든 모델에서 틀린 케이스의 word_index: {2, 454, 6, 335, 17, 276, 117, 57, 218, 446}


word_index: 2

word는 seat이고, 의자가 놓여있는 회화 그림. 정답 센스는 의자 가구를 뜻하는 센스 3. 이 센스가 가장 직접적으로 그림을 표현한다.

---

* gemma-3: web content 있음. 요약문에서 seat을 의자의 일부분인 '좌석'의 의미로 해석. 이에 따라 모델도 seat를 의자가 아닌 좌석으로 판별.
* qwen3: web content 있음. 요약문에서 단어 chair와 seat를 구분해서 사용. seat는 의자 자체가 아닌 좌석의 의미로 사용. 문장은 모호함. 이에 따라 모델은 seat를 의자라는 가구가 아닌 좌석을 뜻하는 4번 sense로 결정.
* mistral-small-3.1: web content 있음. 요약문에서 단어 chair와 seat를 구분해서 사용. seat는 의자 자체가 아닌 좌석의 의미로 사용. 문장은 모호함. 이에 따라 모델은 seat를 의자라는 가구가 아닌 좌석을 뜻하는 4번 sense로 결정.

RAG content에서 seat라는 단어를 사실상 '의자'라는 의미를 배제하고, '좌석'이라는 의미로 사용하며 강한 선입관을 부여. RAG이 결정을 방해

word_index: 6

word는 tympanum이고, 귀의 외이도를 나타낸 그림. 정답 센스는 고막보다는 고실 전체을 뜻하는 센스 1. 이 센스가 가장 직접적으로 그림을 표현한다.

asen: After the flight, my tympanum felt a little blocked.

---

* gemma-3: web content 있음. 요약문은 특정 sense와 직접 관련이 있다기보다는 배경 지식을 이야기함. 해당 배경지식이 의도한 센스가 아닌 2번 고막 센스에 더 가까운 context를 제공. 주어진 asen이 오답인 2번 sense를 너무 강하게 암시함.
* qwen3: web content 있음. 요약문은 특정 sense와 직접 관련이 있다기보다는 전체적인 배경 지식을 이야기함. 해당 배경지식이 의도한 센스가 아닌 2번 고막 센스에 더 가까운 context를 제공. asen이 오답인 2번 sense를 너무 강하게 암시함.
* mistral-small-3.1: web content 있음. 요약문은 특정 sense와 직접 관련이 있다기보다는 전체적인 배경 지식을 이야기함. 해당 배경지식이 의도한 센스가 아닌 2번 고막 센스에 더 가까운 context를 제공. asen이 오답인 2번 sense를 너무 강하게 암시함.

asen이 주는 강한 선입관이 예측을 방해. RAG은 별로 도움을 주지 못함.

word_index: 17

word는 metal이고, 계단 금속 질감을 클로즈업한 사진. 정답 합금을 뜻하는 센스 2.

asen: The jeweler recommended a metal for the bracelet that would keep its color.

---

* gemma-3: web content 없음. asen만 보고 어째서인지 추론 과정에서 합금보다 금속류 전체가 정답이라고 판단. context 부족. asen의 metal은 모호하며, 이미지가 합금이라는 사실을 알려주는 context가 없음
* qwen3: web content 없음. context 부족. asen의 metal은 모호하며, 이미지가 합금이라는 사실을 알려주는 context가 없음
* mistral-small-3.1: web content 있음. context 부족. asen의 metal은 모호하며, 이미지가 합금이라는 사실을 알려주는 context가 없음

context 부족. 이거는 RAG content가 이미지가 보여주는 것이 합금이라는 사실을 확인해 주어야 하는데 RAG content가 하나도 인용되지 않음.

word_index: 57

word는 try, 법원에서 판사가 망치 휘두르는 사진. 정답은 5.

asen: They will try the matter in court tomorrow.

---

* gemma-3: RAG content 있음. 헷갈리는 두 센스 중 어느 쪽이 이미지에 가장 적합한지 알려주는 context가 없음. asen은 모호.
* qwen3: RAG content 있음. 헷갈리는 두 센스 중 어느 쪽이 이미지에 가장 적합한지 알려주는 context가 없음. asen은 모호.
* mistral-small-3.1: RAG content 있음. 헷갈리는 두 센스 중 어느 쪽이 이미지에 가장 적합한지 알려주는 context가 없음. asen은 모호.

이미지에 가장 직접적인 것은 사건을 심리하는 action만을 뜻하는 5번이지만, asen이 주는 강한 선입관이 이미지와 align되어 '판사로서 사건을 심리하다'인 3번 sense를 강하게 push해버림. 가장 특정한 sense를 고르라는 지시 사항을 지키지 못함.

RAG이 포괄적인 재판에 관해 논의하며 sense를 특정한 action을 나타내는 5번보다 포괄적인 재판 주재를 의미하는 3번으로 기울어지게 만듬. RAG이 방해.

word_index: 117

word는 paper, 신문지가 가판대에 쌓여있는 사진. 정답은 7.

asen: I grabbed the paper before heading out.

---

* gemma-3: RAG content는 언론의 자유에 관한 포괄적인 뒷배경. 모델은 더 general한 sense인 3번을 택했지만, prompt에서는 가장 좁은 특정한 의미(specific)를 고르라고 하였으므로 7번이 정답. 프롬프트의 가장 특정한 것을 고르라는 말을 까먹은 것.
* qwen3: RAG content는 언론의 자유에 관한 포괄적인 뒷배경.추론 과정 없음. 그냥 3번.
* mistral-small-3.1: RAG content는 언론의 자유에 관한 포괄적인 뒷배경. RAG의 오류. 언론의 자유에 관한 내용인 RAG 내용이 sense를 신문지라는 좁은 sense보다 언론이라는 포괄적인 sesne에 더 가깝게 위치시킴

RAG 내용이 신문지보다는 포괄적인 언론에 관한 내용이라 3번에 더 sense를 가깝게 보냄. 가장 specific한 의미는 7번이지만, RAG이 방해함..

word_index: 218

word는 caper, 식물의 생태를 묘사한 그림. 정답은 1.

asen: At the market, I picked up a caper.

---

* gemma-3: RAG content는 caper가 식용으로 쓰이는 식물이라는 점을 강조. 모델은 이미지 context를 넘어 RAG을 따라 식용을 꽃봉우리를 뜻하는 2번 sense에 꽂힘.
* qwen3: RAG content는 caper가 식용으로 쓰이는 식물이라는 점을 강조. 모델은 이미지 context를 넘어 RAG을 따라 식용을 꽃봉우리를 뜻하는 2번 sense에 꽂힘.
* mistral-small-3.1: RAG content는 caper가 식용으로 쓰이는 식물이라는 점을 강조. 모델은 이미지 context를 넘어 RAG을 따라 식용을 꽃봉우리를 뜻하는 2번 sense에 꽂힘.

RAG 내용이 이미지에 대한 불필요한 background를 제공하여 정답에서 멀어지게 만듦.

word_index: 276

word는 earthnut, 암석처럼 보이지만 화이트 트러플의 사진. 정답은 4.

asen: At the farmers’ market, I bought an earthnut for dinner.

---

* gemma-3: RAG content는 사진이 화이트 트러플이며 earthnut은 구어체 명칭일 것이라고 추측. 모델은 이미지 context를 넘어 RAG을 따라 식용을 꽃봉우리를 뜻하는 2번 sense에 꽂힘.
* qwen3: 
* mistral-small-3.1: 

RAG 내용이 이미지에 대한 불필요한 background를 제공하여 정답에서 멀어지게 만듦.

In [ ]:
# 각 모델에서 틀린 케이스

